# BehaVERT quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShinSeungJ/BehaVERT/blob/main/notebooks/quickstart.ipynb)

BehaVERT reads animal pose keypoints as a sequence of tokens and decodes behavior from them
([Shin et al., IJCV 2026](https://doi.org/10.1007/s11263-026-02834-y)). This notebook walks through:

1. **Video → behavior** — an open-field video, through a pose model, to per-frame behaviors
2. **Synchronized video** — keypoints, embedding map, attention and predictions side by side (CalMS21)
3. **Your own keypoints** — the input format and the Python API
4. **Dataset-level analysis** *(optional, downloads 3.6 GB)* — attention statistics and embedding clusters
5. **Training on your own data** — where to start

Pretrained models download automatically from [Hugging Face](https://huggingface.co/ShinSeungJ/behavert).
A GPU makes it much faster (Colab: *Runtime → Change runtime type → GPU*).

## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("BehaVERT"):
        !git clone -q https://github.com/ShinSeungJ/BehaVERT
    %cd BehaVERT
    !pip install -q -e ".[pose]"
else:
    # running locally: work from the repository root
    while not os.path.exists("pyproject.toml") and os.getcwd() != "/":
        os.chdir("..")
print("working directory:", os.getcwd())

BEHAVERT = f"{sys.executable} -m behavert"   # the `behavert` command
OUT = "outputs/notebook"

In [ ]:
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Video, Image, display

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be slow)")

## 1. Video → behavior

`behavert decode` runs the pose model `oft-pose` on every frame, feeds the keypoints to the behavior model
`oft-behavior`, and writes the results. `--overlay` adds a video with the skeleton and behavior drawn on.

In [ ]:
!{BEHAVERT} decode --video demo_data/video/oft.mp4 --model oft-behavior --overlay --out {OUT}

Each video gets a folder with `behavior.csv` (one row per frame: behavior, confidence, attention and the
probability of every class), `summary.csv` (time and episodes per behavior), `keypoints.csv` and `overlay.mp4`.

In [ ]:
d = f"{OUT}/oft"
display(pd.read_csv(f"{d}/summary.csv"))
behavior = pd.read_csv(f"{d}/behavior.csv")
behavior.head()

In [ ]:
# ethogram: which behavior, when (same colors as the videos)
from behavert.decode.render import behavior_colors

names = [c[2:] for c in behavior.columns if c.startswith("p_")]
colors = behavior_colors(names)
fig, ax = plt.subplots(figsize=(12, 2.2))
for i, (n, col) in enumerate(zip(names, colors)):
    on = behavior.behavior.values == n
    ax.fill_between(behavior.time_s, i - 0.4, i + 0.4, where=on, step="mid", lw=0, color=col)
ax.set_yticks(range(len(names)), names); ax.set_xlabel("time (s)"); ax.set_xlim(0, behavior.time_s.max())
plt.tight_layout(); plt.show()

In [ ]:
Video(f"{d}/overlay.mp4", embed=True, width=640)

## 2. Synchronized video

With `--sync`, decode also renders the synchronized panel video: the keypoints on the video, every
window's embedding on a t-SNE map (the current one circled), and the attention, confidence, ground truth and
prediction flowing past a fixed cursor.

Here the input is keypoints instead of a video: the first 5 minutes of a CalMS21 test session, with its human
labels (`--labels`). We render the first 30 seconds.

In [ ]:
!{BEHAVERT} decode --keypoints demo_data/keypoints/calms21_mouse071_task1_annotator1.csv \
    --labels demo_data/keypoints/calms21_mouse071_task1_annotator1_labels.csv \
    --video demo_data/video/calms21_mouse071_task1_annotator1.mp4 \
    --model calms21-task1 --sync --frames 900 --out {OUT}

In [ ]:
Video(f"{OUT}/calms21_mouse071_task1_annotator1/sync.mp4", embed=True, width=900)

The same works for the MABe22 strain model (`--keypoints demo_data/keypoints/mabe22_E21WHFXBDGA4XVHZQKD5.csv
--model mabe22-strain`). With the MABe22 dataset downloaded, `--config configs/mabe22/state.yaml --task strain
--recording E21WHFXBDGA4XVHZQKD5` adds a strip of annotated behaviors and a backdrop of other videos to the
embedding map.

## 3. Your own keypoints

A model needs the keypoints it was trained on, in its order, flattened per frame as
animal → keypoint → (x, y). The layout is stored with every model:

In [ ]:
from behavert import BehaVERT
from behavert.decode import KeypointLayout

model = BehaVERT.from_pretrained("oft-behavior")
layout = KeypointLayout.from_config(model.config)
print("animals:", layout.num_animals, "| units:", layout.units)
print("columns:", layout.column_names())
print("classes:", model.config.label_names)

A CSV with these columns (one row per frame; extra `frame` / `time` columns are ignored) can be decoded directly:

```bash
behavert decode --keypoints my_keypoints.csv --video my_video.mp4 --model oft-behavior --overlay --sync
```

or from Python — here with the keypoints that section 1 produced:

In [ ]:
from behavert.inference import predict_recording

keypoints = pd.read_csv(f"{OUT}/oft/keypoints.csv").values        # [frames, 22]
pred = predict_recording(model, keypoints)
print(pred.probs.shape, "per-frame probabilities")
pd.Series(np.array(model.config.label_names)[pred.labels]).value_counts()

## 4. Dataset-level analysis *(optional)*

These commands run over every recording of a benchmark, so they need its data (`behavert download`).
Set `RUN = True` to download MABe22 (3.6 GB) and run them.

- `behavert attention` — per-frame attention with confidence, correctness and behavior labels, and tests of how
  they relate (median splits, behavior co-occurrence), as CSV tables.
- `behavert embed` — a t-SNE / UMAP map of the embeddings colored by class, and clustering scores (NMI, purity).

In [ ]:
RUN = False
if RUN:
    !{BEHAVERT} download mabe22
    !{BEHAVERT} attention configs/mabe22/state.yaml mabe22-strain --task strain --recordings 100 --out {OUT}/attention
    !{BEHAVERT} embed configs/mabe22/state.yaml mabe22-strain --task strain --recordings 200 --out {OUT}/embedding

In [ ]:
if RUN:
    tests = pd.read_csv(f"{OUT}/attention/attention_tests.csv")
    display(tests[["analysis", "group_a", "group_b", "mean_a", "mean_b", "p_value"]])
    w = pd.read_csv(f"{OUT}/attention/attention_windows.csv")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(w.attention_var, w.confidence, s=4, alpha=0.4)
    ax.set_xscale("log"); ax.set_xlabel("attention variance in window (peakedness)"); ax.set_ylabel("confidence")
    plt.tight_layout(); plt.show()
    display(pd.read_csv(f"{OUT}/embedding/clustering.csv"))
    display(Image(f"{OUT}/embedding/embedding.png", width=450))

## 5. Training on your own data

`configs/custom_token.yaml` (one behavior label per frame) and `configs/custom_state.yaml` (one label per
recording, such as a genotype) are commented templates: point one at a folder of CSVs (keypoint columns plus a
`label` column), set your classes and arena size, and run

```bash
behavert train configs/custom_token.yaml --data-root /path/to/your/data
```

The configs for every experiment of the paper are in `configs/` (e.g.
`behavert train configs/mabe22/behavior.yaml --task chase` after `behavert download mabe22`).

In [ ]:
print(open("configs/custom_token.yaml").read()[:1600])